In [0]:
df_csv = spark.read.table("restaurant_project.bronze.raw_sales_csv")
df_json = spark.read.table("restaurant_project.bronze.raw_sales_json")

In [0]:
df_csv.display()

In [0]:
display(df_json)

In [0]:
df_csv.printSchema()

In [0]:
display(df_csv.describe())


In [0]:
df_combine = df_csv.unionByName(df_json)

In [0]:
display(df_combine.limit(5))

In [0]:
from pyspark.sql.functions import col
df_duplicat = df_combine.groupBy("order_id")\
    .count()\
    .filter(col("count")> 1)\
    .orderBy(col("count").desc())
display(df_duplicat.limit(10))

In [0]:
from pyspark.sql.functions import col

# عدد الصفوف اللي فيها الـ order_id بـ null
null_count = df_combine.filter(col("order_id").isNull()).count()

print(f"Total Null IDs: {null_count}")

In [0]:
df_duplicat = df_combine.groupBy("order_date")\
    .count()\
    .filter(col("count")> 1)\
    .orderBy(col("count").desc())
display(df_duplicat.limit(10))

In [0]:
null_count = df_combine.filter(col("order_date").isNull()).count()

print(f"Total Null IDs: {null_count}")

In [0]:
null_count = df_combine.filter(col("hour").isNull()).count()

print(f"Total Null IDs: {null_count}")

In [0]:
null_count = df_combine.filter(col("category").isNull()).count()

print(f"Total Null IDs: {null_count}")

In [0]:
null_count = df_combine.filter(col("item_name").isNull()).count()

print(f"Total Null IDs: {null_count}")

In [0]:
uniqe_cateqory = df_combine.select("category").distinct()
display(uniqe_cateqory)

In [0]:
uniqe_item = df_combine.select("item_name").distinct()
display(uniqe_item)

In [0]:
from pyspark.sql.functions import col


invalid_prices_df = df_combine.filter(
    (col("price").isNull()) | 
    (col("price") <= 0)
)


print(f"عدد السجلات التي تحتوي على أسعار غير منطقية: {invalid_prices_df.count()}")
display(invalid_prices_df)

In [0]:
df_rest_sales = spark.table("restaurant_project.silver.rest_sales")

In [0]:
display(df_rest_sales)

In [0]:
from pyspark.sql import functions as F

# عمل Group By بالعمود الجديد وحساب العدد
weekend_stats = df_rest_sales.groupBy("is_weekend") \
    .agg(F.count("*").alias("total_orders")) \
    .orderBy(F.col("total_orders").desc())

# عرض النتيجة
display(weekend_stats)